In [2]:
import pickle
from elasticsearch import Elasticsearch
import os

from workers.utils.task_definitions import (
    TermDistribution,
    WordCloud,
    DocsRelatedFinder,
    SocialNetwork
)

INDEX_NAME = os.getenv("INDEX_NAME")
es_client = Elasticsearch('http://localhost:9200')

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-28 23:26:28.686 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [ ]:
test_task_instance = TermDistribution(
    min_date="01-01-1999",
    max_date="31-12-2002",
    terms="poland,russia",
)

test_task_name = "term_distribution"

with open(f"stored_instances/{test_task_name}.pkl", "wb") as f:
    pickle.dump(test_task_instance, f)

In [2]:
test_task_instance = WordCloud(
    min_date="01-01-2001",
    max_date="31-12-2002",
    pos_filter="all"
)

test_task_name = "word_cloud"

with open(f"stored_instances/{test_task_name}.pkl", "wb") as f:
    pickle.dump(test_task_instance, f)

In [2]:
test_task_instance = SocialNetwork(
    min_date="01-01-1999",
    max_date="31-12-2002",
    terms="poland,russia",
    window_size=5,
    minimum_edge_weight=1
)

test_task_name = "social_network"

with open(f"stored_instances/{test_task_name}.pkl", "wb") as f:
    pickle.dump(test_task_instance, f)

In [3]:
INDEX_NAME = f"{os.getenv('INDEX_NAME')}_fragments"

In [4]:
query = {
    "bool": {
        "must": [
            {
                "range": {
                    "date": {
                        "gte": "1999-01-01",
                        "lte": "2002-12-31"
                    }
                }
            },
            {
                "bool": {
                    "should": [{"match": {"text": "poland"}}, {"match": {"text": "russia"}}],
                    "minimum_should_match": 1
                }
            }
        ]
    }
}
response = es_client.search(
    index=INDEX_NAME,
    query=query,
    size=10000
)

subset = response["hits"]["hits"]

In [ ]:
rep = es_client.search(index=INDEX_NAME, query={"match_all": {}}, size=100)["hits"]["hits"]

In [5]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

search_input = "military operations and national security"
query_embedding = embed_model.encode(search_input, normalize_embeddings=True).tolist()

response = es_client.search(
    index=os.getenv("INDEX_NAME"),
    query={
        "more_like_this": {
            "fields": ["text", "title"],
            "like": search_input,
            "min_term_freq": 1,
            "min_doc_freq": 1,
        }
    },
    knn={
        "field": "title_embedding",
        "query_vector": query_embedding,
        "k": 10,
        "num_candidates": 50,
    },
    size=10,
)

for hit in response["hits"]["hits"]:
    print(f"{hit['_score']:.4f} | {hit['_source']['title']}")

6.9254 | Excerpts from an Interview with Journalists
5.1055 | Interview with ORT Channel
5.0005 | Introductory Address at a Meeting with Deputy Prime Ministers and Heads of Law-Enforcement Ministries and Departments
4.7979 | Speech at a meeting with top officers promoted to higher positions and specialist military ranks
4.4148 | Speech at the All-Russia Meeting of Defence Industry Workers
3.7265 | Interview with the RTR TV Channel
3.6313 | Address at a Session of the Security Council
3.5452 | Address at a Parade Dedicated to the 55th Anniversary of Victory in the Great Patriotic War
3.4820 | Interview with the ORT TV Channel
3.4775 | News Conference Following Security Council Session


In [7]:
query = {
    "bool": {
        "must": [
            {
                "range": {
                    "date": {
                        "gte": "1999-01-01",
                        "lte": "2002-12-31"
                    }
                }
            },
            {
                "bool": {
                    "should": [{"match": {"text": "poland"}}, {"match": {"text": "russia"}}],
                    "minimum_should_match": 1
                }
            }
        ]
    }
}
response = es_client.search(
    index=INDEX_NAME,
    query=query,
    size=10000
)

subset = response["hits"]["hits"]